# Week 37

In [ ]:
import pandas as pd
import re
import os
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Functions from week 36

In [ ]:
# !pip install camel-tools konlpy indic-nlp-library

In [ ]:
# Remove unwanted characters from the questions
def cleanDf(df):
    pattern = re.compile(r"[?؟,;\/\\\[\]#():]")
    df['question'] = df['question'].apply(lambda x: pattern.sub("", x))
    df['context'] = df['context'].apply(lambda x: pattern.sub("", x))
    return df

df_train_clean = cleanDf(df_train)
df_val_clean = cleanDf(df_val)

In [ ]:
from camel_tools.tokenizers.word import simple_word_tokenize
from konlpy.tag import Okt
from indicnlp.tokenize import indic_tokenize

okt = Okt()

langForStat = ['ar', 'ko', 'te']

numQuestions = []

def tokenize_by_lang(text, lang):
    if not isinstance(text, str):
        return []
    if lang == 'ar':
        return simple_word_tokenize(text)
    elif lang == 'ko':
        return okt.morphs(text)
    elif lang == 'te':
        return indic_tokenize.trivial_tokenize(text)
    else:
        return text.split()

for lang in langForStat:
    num_train = df_train_clean[df_train_clean['lang'] == lang].shape[0]
    num_val = df_val_clean[df_val_clean['lang'] == lang].shape[0]
    numQuestions.append((lang, num_train, num_val))

    df_train_lang = df_train_clean[df_train_clean['lang'] == lang].copy()
    df_val_lang = df_val_clean[df_val_clean['lang'] == lang].copy()

    df_train_lang['tokens'] = df_train_lang['question'].apply(lambda x: tokenize_by_lang(x, lang))
    df_val_lang['tokens'] = df_val_lang['question'].apply(lambda x: tokenize_by_lang(x, lang))

    df_train_lang['wordcount'] = df_train_lang['tokens'].apply(len)

### Week 37

In [ ]:
arabicDf_train = df_train_clean[df_train_clean['lang'] == 'ar'].copy()
teluguDf_train = df_train_clean[df_train_clean['lang'] == 'te'].copy()
koreanDf_train = df_train_clean[df_train_clean['lang'] == 'ko'].copy()

arabicDf_val = df_val_clean[df_val_clean['lang'] == 'ar'].copy()
teluguDf_val = df_val_clean[df_val_clean['lang'] == 'te'].copy()
koreanDf_val = df_val_clean[df_val_clean['lang'] == 'ko'].copy()

In [ ]:
def build_counts(corpus):
    allUnigrams = []
    allBigrams = []
    allTrigrams = []

    for text in corpus:
        tokens = nltk.word_tokenize(text)
        allUnigrams.extend(tokens)
        allBigrams.extend(list(ngrams(tokens, 2)))
        allTrigrams.extend(list(ngrams(tokens, 3)))

    unigram_fd = FreqDist(allUnigrams)
    bigram_fd = FreqDist(allBigrams)
    trigram_fd = FreqDist(allTrigrams)

    return unigram_fd, bigram_fd, trigram_fd

def conditional_prob_unigram(w, unigram_fd):
    return unigram_fd[w] / sum(unigram_fd.values())

def conditional_prob_bigram(w2, w1, bigram_fd, unigram_fd, k=0.0):
    bi = bigram_fd[(w1,w2)]
    uni = unigram_fd[w1]
    if uni>0:
        return bi/uni
    return 1.0/V

def conditional_prob_trigram(w3, w1, w2, trigram_fd, bigram_fd, V, k=0.0):
    tri = trigram_fd[(w1,w2,w3)]
    bi  = bigram_fd[(w1,w2)]
    if bi>0:
        return (tri + k) / (bi + k*V)
    return 1.0/V

def sentence_logprob_interpolated(sentence, unigram_fd, bigram_fd, trigram_fd,
                                  V, lambdas=(0.1,0.3,0.6), k=0.0):
    lam1, lam2, lam3 = lambdas
    toks = nltk.word_tokenize(sentence)
    trigs = list(ngrams(toks, 3))
    logp = 0.0
    for w1,w2,w3 in trigs:
        p_uni = conditional_prob_unigram(w3, unigram_fd)
        p_bi  = conditional_prob_bigram(w3, w2, bigram_fd, unigram_fd, k)
        p_tri = conditional_prob_trigram(w3, w1, w2, trigram_fd, bigram_fd, V, k)
        p = lam1*p_uni + lam2*p_bi + lam3*p_tri
        logp += math.log(p)
    return logp, len(toks)

### Arabic

In [ ]:
# Unigram model
allUnigrams_ar = []

for q in arabicDf_train['question']:
    tokens = simple_word_tokenize(q)
    allUnigrams_ar.extend(tokens)

unigram_fd_ar = FreqDist(allUnigrams_ar)
total_tokens_train = sum(unigram_fd_ar.values())
V = len(unigram_fd_ar)

def question_logprob_unigram(sentence, unigram_fd, total_tokens, V, k=0.1):
    tokens = simple_word_tokenize(sentence)
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        prob = (count + k) / (total_tokens + k * V)
        log_prob += math.log(prob)
    return log_prob, len(tokens)

total_log_prob_ar = 0.0
total_tokens_ar = 0

for q in arabicDf_val['question']:
    logp, n = question_logprob_unigram(q, unigram_fd_ar, total_tokens_train, V)
    total_log_prob_ar += logp
    total_tokens_ar += n

perplexity_uni_ar = math.exp(-total_log_prob_ar / total_tokens_ar)

In [ ]:
# Bigram model
allBigrams_ar, allUnigrams_ar = [], []
for q in arabicDf_train['question']:
    tokens = (['<s>'] + simple_word_tokenize(q) + ['</s>'])
    allBigrams_ar.extend(list(ngrams(tokens, 2)))
    allUnigrams_ar.extend(tokens)

unigram_fd_ar = FreqDist(allUnigrams_ar)
bigram_fd_ar = FreqDist(allBigrams_ar)
V_ar = len(unigram_fd_ar)
total_tokens_train = sum(unigram_fd_ar.values())

def bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k=0.1):
    bi = bigram_fd[(w1, w2)]
    uni = unigram_fd[w1]
    return (bi + k) / (uni + k * V)

def compute_logprob(sentence, unigram_fd, bigram_fd, V, k=0.1):
    tokens = (['<s>'] + simple_word_tokenize(sentence) + ['</s>'])
    if len(tokens) < 2:
        return 0.0, 0
    bigrams = list(ngrams(tokens, 2))
    log_prob = 0.0
    for w1, w2 in bigrams:
        p = bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k)
        log_prob += math.log(p)
    return log_prob, len(tokens)

total_log_prob_ar = 0.0
total_tokens_ar = 0
for q in arabicDf_val['question']:
    logp, n = compute_logprob(q, unigram_fd_ar, bigram_fd_ar, V_ar, k=0.1)
    total_log_prob_ar += logp
    total_tokens_ar += n

perplexity_bi_ar = math.exp(-total_log_prob_ar / total_tokens_ar)


In [ ]:
# Trigram model
allTrigrams_ar, allBigrams_ar, allUnigrams_ar = [], [], []
for q in arabicDf_train['question']:
    tokens = (['<s>', '<s>'] + simple_word_tokenize(q) + ['</s>'])
    allUnigrams_ar.extend(tokens)
    allBigrams_ar.extend(list(ngrams(tokens, 2)))
    allTrigrams_ar.extend(list(ngrams(tokens, 3)))

unigram_fd_ar = FreqDist(allUnigrams_ar)
bigram_fd_ar = FreqDist(allBigrams_ar)
trigram_fd_ar = FreqDist(allTrigrams_ar)

def trigram_prob_addk(w1, w2, w3, trigram_fd, bigram_fd, V, k=0.1):
    tri = trigram_fd[(w1, w2, w3)]
    bi  = bigram_fd[(w1, w2)]
    return (tri + k) / (bi + k * V)

def question_logprob(sentence, trigram_fd, bigram_fd, V, k=0.1):
    tokens = (['<s>', '<s>'] + simple_word_tokenize(sentence) + ['</s>'])
    if len(tokens) < 3:
        return 0.0, 0
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        p = trigram_prob_addk(w1, w2, w3, trigram_fd, bigram_fd, V, k)
        log_prob += math.log(p)
    return log_prob, len(tokens)

total_log_prob_ar = 0.0
total_tokens_ar = 0
for q in arabicDf_val['question']:
    logp, n = question_logprob(q, trigram_fd_ar, bigram_fd_ar, V_ar, k=0.1)
    total_log_prob_ar += logp
    total_tokens_ar += n

perplexity_tri_ar = math.exp(-total_log_prob_ar / total_tokens_ar)


In [ ]:
# Interpolation model
total_log, total_tokens = 0.0, 0
for s in arabicDf_val['question']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd_ar, bigram_fd_ar, trigram_fd_ar, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n

perplexity_inter_ar = math.exp(-total_log/total_tokens)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (Arabic): {perplexity_uni_ar}")
print(f"Bigram Perplexity (Arabic): {perplexity_bi_ar}")
print(f"Trigram Perplexity (Arabic): {perplexity_tri_ar}")
print(f"Interpolated Trigram Perplexity (Arabic): {perplexity_inter_ar}")

Unigram Perplexity (Arabic): 2504.7481430122075
Bigram Perplexity (Arabic): 314.170115058628
Trigram Perplexity (Arabic): 222.7016734506883
Interpolated Trigram Perplexity (Arabic): 144.67832205646965


In [ ]:
# Unseen trigrams
def unseen_rate_tri(sentences, trigram_fd, use_bounds=True):
    unseen, total = 0, 0
    for s in sentences:
        toks = (['<s>', '<s>'] + simple_word_tokenize(s) + ['</s>']) if use_bounds else simple_word_tokenize(s)
        for w1, w2, w3 in nltk.ngrams(toks, 3):
            total += 1
            unseen += (trigram_fd[(w1, w2, w3)] == 0)
    return unseen/total if total else 0.0

val_sentences = arabicDf_val['question'].tolist()

tri_unseen = unseen_rate_tri(val_sentences, trigram_fd_ar)

print(f"Trigram unseen rate: {tri_unseen}")

Trigram unseen rate: 0.641211323238973%


### Korean

In [ ]:
# Unigram model
allUnigrams_ko = []

okt = Okt()

for q in koreanDf_train['question']:
    tokens = okt.morphs(q)
    allUnigrams_ko.extend(tokens)

unigram_fd_ko = FreqDist(allUnigrams_ko)
total_tokens_train = sum(unigram_fd_ko.values())
V_ko = len(unigram_fd_ko)

def question_logprob_unigram(sentence, unigram_fd, total_tokens, V_ko, k=0.1):
    tokens = okt.morphs(sentence)
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        prob = (count + k) / (total_tokens + k * V_ko)
        log_prob += math.log(prob)
    return log_prob, len(tokens)

total_log_prob_ko = 0.0
total_tokens_ko = 0

for q in koreanDf_val['question']:
    logp, n = question_logprob_unigram(q, unigram_fd_ko, total_tokens_train, V_ko)
    total_log_prob_ko += logp
    total_tokens_ko += n

perplexity_uni_ko = math.exp(-total_log_prob_ko / total_tokens_ko)

In [ ]:
# Bigram model
allBigrams_ko, allUnigrams_ko = [], []
for q in koreanDf_train['question']:
    tokens = (['<s>'] + okt.morphs(q) + ['</s>'])
    allBigrams_ko.extend(list(ngrams(tokens, 2)))
    allUnigrams_ko.extend(tokens)

unigram_fd_ko = FreqDist(allUnigrams_ko)
bigram_fd_ko = FreqDist(allBigrams_ko)
V_ko = len(unigram_fd_ko)
total_tokens_train = sum(unigram_fd_ko.values())

def bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k=0.1):
    bi = bigram_fd[(w1, w2)]
    uni = unigram_fd[w1]
    return (bi + k) / (uni + k * V)

def compute_logprob(sentence, unigram_fd, bigram_fd, V, k=0.1):
    tokens = (['<s>'] + okt.morphs(sentence) + ['</s>'])
    if len(tokens) < 2:
        return 0.0, 0
    bigrams = list(ngrams(tokens, 2))
    log_prob = 0.0
    for w1, w2 in bigrams:
        p = bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k)
        log_prob += math.log(p)
    return log_prob, len(tokens)

total_log_prob_ko = 0.0
total_tokens_ko = 0
for q in koreanDf_val['question']:
    logp, n = compute_logprob(q, unigram_fd_ko, bigram_fd_ko, V_ko, k=0.1)
    total_log_prob_ko += logp
    total_tokens_ko += n

perplexity_bi_ko = math.exp(-total_log_prob_ko / total_tokens_ko)


In [ ]:
# Trigram model
allTrigrams_ko, allBigrams_ko, allUnigrams_ko = [], [], []
for q in koreanDf_train['question']:
    tokens = ['<s>', '<s>'] + okt.morphs(q) + ['</s>']
    allUnigrams_ko.extend(tokens)
    allBigrams_ko.extend(list(ngrams(tokens, 2)))
    allTrigrams_ko.extend(list(ngrams(tokens, 3)))

unigram_fd_ko = FreqDist(allUnigrams_ko)
bigram_fd_ko = FreqDist(allBigrams_ko)
trigram_fd_ko = FreqDist(allTrigrams_ko)

V_ko = len(unigram_fd_ko)

def trigram_prob_addk(w1, w2, w3, trigram_fd, bigram_fd, V, k=0.1):
    tri = trigram_fd[(w1, w2, w3)]
    bi  = bigram_fd[(w1, w2)]
    return (tri + k) / (bi + k * V)

def question_logprob(sentence, trigram_fd, bigram_fd, V, k=0.1):
    tokens = (['<s>', '<s>'] + okt.morphs(sentence) + ['</s>'])
    if len(tokens) < 3:
        return 0.0, 0
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        p = trigram_prob_addk(w1, w2, w3, trigram_fd, bigram_fd, V, k)
        log_prob += math.log(p)
    return log_prob, len(tokens)

total_log_prob_ko = 0.0
total_tokens_ko = 0
for q in koreanDf_val['question']:
    logp, n = question_logprob(q, trigram_fd_ko, bigram_fd_ko, V_ko, k=0.1)
    total_log_prob_ko += logp
    total_tokens_ko += n

perplexity_tri_ko = math.exp(-total_log_prob_ko / total_tokens_ko)


In [ ]:
# Interpolation model
total_log, total_tokens = 0.0, 0
for s in koreanDf_val['question']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd_ko, bigram_fd_ko, trigram_fd_ko, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n

perplexity_inter_ko = math.exp(-total_log/total_tokens)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (Korean): {perplexity_uni_ko}")
print(f"Bigram Perplexity (Korean): {perplexity_bi_ko}")
print(f"Trigram Perplexity (Korean): {perplexity_tri_ko}")
print(f"Interpolated Trigram Perplexity (Korean): {perplexity_inter_ko}")

Unigram Perplexity (Korean): 506.6485506020069
Bigram Perplexity (Korean): 90.83640629546088
Trigram Perplexity (Korean): 133.05586789093326
Interpolated Trigram Perplexity (Korean): 109.75377521122746


In [ ]:
# Unseen trigrams
def unseen_rate_tri(sentences, trigram_fd, use_bounds=True):
    unseen, total = 0, 0
    for s in sentences:
        toks = (['<s>', '<s>'] + okt.morphs(s) + ['</s>']) if use_bounds else okt.morphs(s)
        for w1, w2, w3 in nltk.ngrams(toks, 3):
            total += 1
            unseen += (trigram_fd[(w1, w2, w3)] == 0)
    return unseen/total if total else 0.0

val_sentences = koreanDf_val['question'].tolist()

tri_unseen = unseen_rate_tri(val_sentences, trigram_fd_ko)

print(f"Trigram unseen rate: {tri_unseen}")


Trigram unseen rate: 0.47399199753770394


### Telugu

In [ ]:
# Unigram model
allUnigrams_te = []

for q in teluguDf_train['question']:
    tokens = indic_tokenize.trivial_tokenize(q)
    allUnigrams_te.extend(tokens)

unigram_fd_te = FreqDist(allUnigrams_te)
total_tokens_train = sum(unigram_fd_te.values())
V_te = len(unigram_fd_te)

def question_logprob_unigram(sentence, unigram_fd, total_tokens, V_te, k=0.1):
    tokens = indic_tokenize.trivial_tokenize(sentence)
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        prob = (count + k) / (total_tokens + k * V_te)
        log_prob += math.log(prob)
    return log_prob, len(tokens)

total_log_prob_te = 0.0
total_tokens_te = 0

for q in teluguDf_val['question']:
    logp, n = question_logprob_unigram(q, unigram_fd_te, total_tokens_train, V_te)
    total_log_prob_te += logp
    total_tokens_te += n

perplexity_uni_te = math.exp(-total_log_prob_te / total_tokens_te)

In [ ]:
# Bigram model
allBigrams_te, allUnigrams_te = [], []
for q in teluguDf_train['question']:
    tokens = (['<s>'] + indic_tokenize.trivial_tokenize(q) + ['</s>'])
    allBigrams_te.extend(list(ngrams(tokens, 2)))
    allUnigrams_te.extend(tokens)

unigram_fd_te = FreqDist(allUnigrams_te)
bigram_fd_te = FreqDist(allBigrams_te)
V_te = len(unigram_fd_te)
total_tokens_train = sum(unigram_fd_te.values())

def bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k=0.1):
    bi = bigram_fd[(w1, w2)]
    uni = unigram_fd[w1]
    return (bi + k) / (uni + k * V)

def compute_logprob(sentence, unigram_fd, bigram_fd, V, k=0.1):
    tokens = (['<s>'] + indic_tokenize.trivial_tokenize(sentence) + ['</s>'])
    if len(tokens) < 2:
        return 0.0, 0
    bigrams = list(ngrams(tokens, 2))
    log_prob = 0.0
    for w1, w2 in bigrams:
        p = bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k)
        log_prob += math.log(p)
    return log_prob, len(tokens)

total_log_prob_te = 0.0
total_tokens_te = 0
for q in teluguDf_val['question']:
    logp, n = compute_logprob(q, unigram_fd_te, bigram_fd_te, V_te, k=0.1)
    total_log_prob_te += logp
    total_tokens_te += n

perplexity_bi_te = math.exp(-total_log_prob_te / total_tokens_te)

In [ ]:
# Trigram model
allTrigrams_te, allBigrams_te, allUnigrams_te = [], [], []
for q in teluguDf_train['question']:
    tokens = (['<s>', '<s>'] + indic_tokenize.trivial_tokenize(q) + ['</s>'])
    allUnigrams_te.extend(tokens)
    allBigrams_te.extend(list(ngrams(tokens, 2)))
    allTrigrams_te.extend(list(ngrams(tokens, 3)))

unigram_fd_te = FreqDist(allUnigrams_te)
bigram_fd_te = FreqDist(allBigrams_te)
trigram_fd_te = FreqDist(allTrigrams_te)

def trigram_prob_addk(w1, w2, w3, trigram_fd, bigram_fd, V, k=0.1):
    tri = trigram_fd[(w1, w2, w3)]
    bi  = bigram_fd[(w1, w2)]
    return (tri + k) / (bi + k * V)

def question_logprob(sentence, trigram_fd, bigram_fd, V, k=0.1):
    tokens = (['<s>', '<s>'] + indic_tokenize.trivial_tokenize(sentence) + ['</s>'])
    if len(tokens) < 3:
        return 0.0, 0
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        p = trigram_prob_addk(w1, w2, w3, trigram_fd, bigram_fd, V, k)
        log_prob += math.log(p)
    return log_prob, len(tokens)

total_log_prob_te = 0.0
total_tokens_te = 0
for q in teluguDf_val['question']:
    logp, n = question_logprob(q, trigram_fd_te, bigram_fd_te, V_te, k=0.1)
    total_log_prob_te += logp
    total_tokens_te += n

perplexity_tri_te = math.exp(-total_log_prob_te / total_tokens_te)


In [ ]:
# Interpolation model
total_log, total_tokens = 0.0, 0
for s in teluguDf_val['question']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd_te, bigram_fd_te, trigram_fd_te, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n
perplexity_inter_te = math.exp(-total_log/total_tokens)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (Telugu): {perplexity_uni_te}")
print(f"Bigram Perplexity (Telugu): {perplexity_bi_te}")
print(f"Trigram Perplexity (Telugu): {perplexity_tri_te}")
print(f"Interpolated Trigram Perplexity (Telugu): {perplexity_inter_te}")

Unigram Perplexity (Telugu): 1557.709133634498
Bigram Perplexity (Telugu): 170.5614067576682
Trigram Perplexity (Telugu): 176.9750330268949
Interpolated Trigram Perplexity (Telugu): 34.403781447646686


In [ ]:
# Unseen trigrams
def unseen_rate_tri(sentences, trigram_fd, use_bounds=True):
    unseen, total = 0, 0
    for s in sentences:
        toks = (['<s>', '<s>'] + indic_tokenize.trivial_tokenize(s) + ['</s>']) if use_bounds else indic_tokenize.trivial_tokenize(s)
        for w1, w2, w3 in nltk.ngrams(toks, 3):
            total += 1
            unseen += (trigram_fd[(w1, w2, w3)] == 0)
    return unseen/total if total else 0.0

val_sentences = teluguDf_val['question'].tolist()

tri_unseen = unseen_rate_tri(val_sentences, trigram_fd_te)

print(f"Trigram unseen rate: {tri_unseen}")

Trigram unseen rate: 0.5654275092936804


### English

In [ ]:
# Unigram model
allUnigrams_en = []

for q in df_train['context']:
    tokens = nltk.word_tokenize(q.lower())
    allUnigrams_en.extend(tokens)

unigram_fd_en = FreqDist(allUnigrams_en)
total_tokens_train = sum(unigram_fd_en.values())
V_en = len(unigram_fd_en)

def question_logprob_unigram(sentence, unigram_fd, total_tokens, V_en, k=0.1):
    tokens = nltk.word_tokenize(sentence.lower())
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        prob = (count + k) / (total_tokens + k * V_en)
        log_prob += math.log(prob)
    return log_prob, len(tokens)

total_log_prob_en = 0.0
total_tokens_en = 0

for q in df_val['context']:
    logp, n = question_logprob_unigram(q, unigram_fd_en, total_tokens_train, V_en)
    total_log_prob_en += logp
    total_tokens_en += n

perplexity_uni_en = math.exp(-total_log_prob_en / total_tokens_en)

In [ ]:
# Bigram model
allBigrams_en, allUnigrams_en = [], []
for q in df_train['context']:
    tokens = (['<s>'] + nltk.word_tokenize(q.lower()) + ['</s>'])
    allBigrams_en.extend(list(ngrams(tokens, 2)))
    allUnigrams_en.extend(tokens)

unigram_fd_en = FreqDist(allUnigrams_en)
bigram_fd_en = FreqDist(allBigrams_en)
V_en = len(unigram_fd_en)
total_tokens_train = sum(unigram_fd_en.values())

def bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k=0.1):
    bi = bigram_fd[(w1, w2)]
    uni = unigram_fd[w1]
    return (bi + k) / (uni + k * V)

def compute_logprob(sentence, unigram_fd, bigram_fd, V, k=0.1):
    tokens = (['<s>'] + nltk.word_tokenize(sentence.lower()) + ['</s>'])
    if len(tokens) < 2:
        return 0.0, 0
    bigrams = list(ngrams(tokens, 2))
    log_prob = 0.0
    for w1, w2 in bigrams:
        p = bigram_prob_addk(w1, w2, bigram_fd, unigram_fd, V, k)
        log_prob += math.log(p)
    return log_prob, len(tokens)

total_log_prob_en = 0.0
total_tokens_en = 0
for q in df_val['context']:
    logp, n = compute_logprob(q, unigram_fd_en, bigram_fd_en, V_en, k=0.1)
    total_log_prob_en += logp
    total_tokens_en += n

perplexity_bi_en = math.exp(-total_log_prob_en / total_tokens_en)

In [ ]:
# Trigram model
allTrigrams_en = []
allBigrams_en = []
allUnigrams_en = []

for q in df_train['context']:
    tokens = ["<s>", "<s>"] + nltk.word_tokenize(q.lower()) + ["</s>"]
    allUnigrams_en.extend(tokens)
    allBigrams_en.extend(list(ngrams(tokens, 2)))
    allTrigrams_en.extend(list(ngrams(tokens, 3)))

unigram_fd_en = FreqDist(allUnigrams_en)
bigram_fd_en = FreqDist(allBigrams_en)
trigram_fd_en = FreqDist(allTrigrams_en)

def question_logprob(sentence, bigram_fd, trigram_fd, smoothing=1e-6):
    tokens = ["<s>", "<s>"] + nltk.word_tokenize(sentence.lower()) + ["</s>"]
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        trigram_count = trigram_fd[(w1, w2, w3)]
        bigram_count = bigram_fd[(w1, w2)]
        if bigram_count > 0 and trigram_count > 0:
            prob = trigram_count / bigram_count
        else:
            prob = smoothing
        log_prob += math.log(prob)
    return log_prob, len(tokens)

total_log_prob_en = 0.0
total_tokens_en = 0

for q in df_val['context']:
    logp, n = question_logprob(q, bigram_fd_en, trigram_fd_en)
    total_log_prob_en += logp
    total_tokens_en += n

perplexity_tri_en = math.exp(-total_log_prob_en / total_tokens_en)

In [ ]:
# Interpolation model
total_log, total_tokens = 0.0, 0
for s in df_val['context']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd_en, bigram_fd_en, trigram_fd_en, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n
perplexity_inter_en = math.exp(-total_log/total_tokens)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (English): {perplexity_uni_en}")
print(f"Bigram Perplexity (English): {perplexity_bi_en}")
print(f"Trigram Perplexity (English): {perplexity_tri_en}")
print(f"Interpolated Trigram Perplexity (English): {perplexity_inter_en}")

Unigram Perplexity (English): 1863.700341007377
Bigram Perplexity (English): 2498.6126637205048
Trigram Perplexity (English): 7577.90766077585
Interpolated Trigram Perplexity (English): 626.0311810737549


In [ ]:
# Unseen trigrams
def unseen_rate_tri(sentences, trigram_fd, use_bounds=True):
    unseen, total = 0, 0
    for s in sentences:
        toks = (['<s>', '<s>'] + nltk.word_tokenize(s.lower()) + ['</s>']) if use_bounds else nltk.word_tokenize(s.lower())
        for w1, w2, w3 in nltk.ngrams(toks, 3):
            total += 1
            unseen += (trigram_fd[(w1, w2, w3)] == 0)
    return unseen/total if total else 0.0

val_sentences = df_val['context'].tolist()

tri_unseen = unseen_rate_tri(val_sentences, trigram_fd_en)

print(f"Trigram unseen rate: {tri_unseen}")


Trigram unseen rate: 0.5902790583317966


### Dataset statistics

In [ ]:
print("Arabic vocabulary size:", len(unigram_fd_ar))
print("Korean vocabulary size:", len(unigram_fd_ko))
print("Telugu vocabulary size:", len(unigram_fd_te))
print("English vocabulary size:", len(unigram_fd_en))

print("Arabic total words:", sum(unigram_fd_ar.values()))
print("Korean total words:", sum(unigram_fd_ko.values()))
print("Telugu total words:", sum(unigram_fd_te.values()))
print("English total words:", sum(unigram_fd_en.values()))

for lang in langForStat + ['en']:
    if lang == 'en':
        train_texts = df_train_clean['context'].dropna().astype(str)
        val_texts = df_val_clean['context'].dropna().astype(str)
    else:
        train_texts = df_train_clean[df_train_clean['lang'] == lang]['question'].dropna().astype(str)
        val_texts = df_val_clean[df_val_clean['lang'] == lang]['question'].dropna().astype(str)

    train_vocab = set()
    for text in train_texts:
        train_vocab.update(text.split())

    val_words = []
    for text in val_texts:
        val_words.extend(text.split())
    if len(val_words) == 0:
        percent = 0.0
    else:
        in_train = sum(1 for w in val_words if w in train_vocab)
        percent = 100 * in_train / len(val_words)
    print(f"{lang}: {percent:.2f}% of validation words are in training vocabulary")

Arabic vocabulary size: 5408
Korean vocabulary size: 3325
Telugu vocabulary size: 2417
English vocabulary size: 83954
Arabic total words: 23974
Korean total words: 27079
Telugu total words: 11802
English total words: 1671296
ar: 76.84% of validation words are in training vocabulary
ko: 71.83% of validation words are in training vocabulary
te: 79.86% of validation words are in training vocabulary
en: 94.62% of validation words are in training vocabulary
